# Student application 2 — Linear regression and optimization

Allow 60–90 minutes. Keep the supplied split, seeds, and checks. The matching [PDF brief](02_linear_regression_and_optimization_application.pdf) contains the same task guidance and submission checklist.

## Problem statement

A service team wants to estimate how many milliseconds a new request will take, before it runs. This time the required output is a numerical duration, not a slow/not-slow label. Investigate whether payload size and the number of waiting jobs help predict that duration.

**What you are given.** The notebook generates 600 fictional past requests, each with payload in MB, queue length, and measured duration in ms. It includes random variation, so even a useful model will make errors. Supplied partitions contain 360 training, 120 validation, and 120 test rows. The generator and split are already written; no dataset collection is required.

**Concrete example.** For an arriving request carrying 4 MB with 3 jobs waiting, the model should return one number in ms. The simulator has a mean duration of 40 + 8*4 + 3*3 = 81 ms for these inputs; a particular observed duration may differ because of noise. The exercise is to estimate a rule from past observations, not to copy the simulator coefficients into a predictor.

**The question to answer.** How much does a fitted two-input linear rule improve on always predicting the average training duration, and how does one gradient update change its coefficients?

### Your assignment

1. Fit a constant training-mean baseline and a linear model using both input columns; compare validation errors in ms.
2. Complete one gradient update on the supplied scaled design matrix and verify that it lowers training error. This small calculation illustrates fitting; it does not replace the fitted model from task 1.
3. Evaluate the fixed two-input linear model and baseline on test rows; interpret the fitted slopes and remaining errors.

**Supplied versus your work.** Data generation, partitions, the gradient-step setup, and score reports are supplied. You fill three task cells; the two-input model family and step size are fixed for this application.

**When you are finished.** Submit both error comparisons, the three updated coefficients from the gradient illustration, test predictions, and explanations of slope units and residual noise. Preserve the declared evaluation protocol; do not tune a new model after seeing test errors.

## How to work through this application

1. Run the supplied imports and data cells first; they create the variables described below.
2. For each task, read **Goal → Inputs → Steps → Commands → Expected result** before editing.
3. The TODO cell contains a starter outline. Replace each `...` with the requested expression and remove the deliberate `raise NotImplementedError(...)` line when ready.
4. Run the following supplied check and inspect its output. If it fails, correct your task code rather than editing the check.
5. Answer the short interpretation prompts, then restart the kernel and run the completed notebook in order.

**Reading object methods:** `object.method(arguments)` calls an operation on that object. Creating an estimator
sets its options; `fit` learns and stores state; `predict`, `predict_proba`, and `transform` apply learned state.
Training labels belong in `fit`; validation/test labels belong in score calculations.

**Shapes:** `(rows, columns)` describes a table/matrix; `(rows,)` describes a one-dimensional array.
`tr` means training, `va` validation, and `te` test. Raw feature tables use `X`; prepared numeric matrices use `Z`.
Class predictions are labels (0 or 1); event probabilities are numbers between 0 and 1.

## Variable dictionary

| Name | Meaning |
|---|---|
| `X` | 600 rows with payload (MB) and queue (number of waiting requests). |
| `y` | Measured latency in ms: 40 + 8*payload + 3*queue + random noise with SD 8 ms. |
| `Xtr, Xva, Xte` | Raw feature tables with 360 training, 120 validation, and 120 test rows; each has two columns. |
| `ytr, yva, yte` | Matching observed latencies for the three tables. These are real numbers, not classes. |
| `mean_baseline, linear` | The two model objects you will fit: a constant training-mean predictor and a linear predictor. |
| `D, target` | Supplied before task 2: D is a (360, 3) design matrix of ones and two scaled training features; target is the length-360 training latency array. |
| `beta, g, beta_next` | Length-3 coefficient vectors ordered intercept, scaled payload slope, scaled queue slope. beta starts at zero; g is the gradient; beta_next is one update. |
| `prediction` | The length-120 array of final test predictions you create in task 3. |

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, log_loss, mean_squared_error, mean_absolute_error)
print('Python', sys.version.split()[0], '| scikit-learn', sklearn.__version__)
plt.rcParams.update({'figure.figsize': (6, 4), 'axes.grid': True, 'grid.alpha': 0.2})

## Supplied data and protocol

The regression simulator is y = 40 + 8*payload + 3*queue + noise, with noise SD 8 ms. Data seed 31, split seed 23; 360/120/120 rows.

In [ ]:
rng = np.random.default_rng(31)
n = 600
X = pd.DataFrame({'payload': rng.uniform(0.5, 10, n), 'queue': rng.poisson(3, n)})
y = 40 + 8*X.payload + 3*X.queue + rng.normal(0, 8, n)
train_valid, test = train_test_split(np.arange(n), test_size=0.2, random_state=23)
train, valid = train_test_split(train_valid, test_size=0.25, random_state=23)
assert not (set(train) & set(valid) or set(train) & set(test) or set(valid) & set(test))
Xtr, Xva, Xte = X.iloc[train], X.iloc[valid], X.iloc[test]
ytr, yva, yte = y.iloc[train], y.iloc[valid], y.iloc[test]
def regression_scores(y_true, prediction):
    return {'MAE_ms': mean_absolute_error(y_true, prediction),
            'RMSE_ms': np.sqrt(mean_squared_error(y_true, prediction))}
print('Rows:', len(train), len(valid), len(test))

## Task 1. Fit two predictors

**Goal:** Create two fitted prediction rules and inspect their validation errors.

**Inputs already available:** Use Xtr and ytr for fitting. The evaluation cell already supplies Xva and yva.

### Steps

1. Create mean_baseline using DummyRegressor(strategy="mean"); fit it on Xtr, ytr.
2. Create linear using LinearRegression(); fit it on the same training rows.
3. Run the supplied validation table. Identify which row is the baseline and record MAE and RMSE in ms.
4. The two-feature linear family is specified in advance for this exercise. Do not add features or choose settings from its test error.

### Commands you will use

| Command | What it does |
|---|---|
| `DummyRegressor(strategy="mean")` | Creates a model that learns the mean of the training targets and predicts that same number for every row. |
| `LinearRegression()` | Creates an ordinary least-squares model with an intercept. The input columns determine which slopes it fits. |
| `model.fit(Xtr, ytr)` | Estimates model parameters using only training examples and stores them in model. |
| `model.predict(Xva)` | Applies fitted coefficients to validation inputs and returns one predicted latency per row. |
| `regression_scores(actual, predicted)` | Supplied helper returning MAE_ms and RMSE_ms. MAE averages absolute errors; RMSE takes the square root of average squared errors. Both are in ms; lower is better. |

**Expected result:** Two fitted model objects named mean_baseline and linear, followed by a two-row validation error table.

**If you get stuck:** Do not pass yva to fit. A successful fit is not a claim of zero error: the generated outcomes include noise.

In [ ]:
# TODO 1: Fit two predictors
# Replace each ... below, then remove the deliberate stop.
raise NotImplementedError("Complete task 1: Fit two predictors")

mean_baseline = DummyRegressor(strategy="mean")
mean_baseline.fit(..., ...)
linear = LinearRegression()
linear.fit(..., ...)

**Supplied check and output:** Two fitted model objects named mean_baseline and linear, followed by a two-row validation error table. Assertions stop execution when a stated condition fails; printed scores still need interpretation.

In [ ]:
display(pd.DataFrame({'mean': regression_scores(yva, mean_baseline.predict(Xva)), 'linear': regression_scores(yva, linear.predict(Xva))}).T)

## Task 2. Complete one gradient step

**Goal:** Translate the half-MSE gradient into array operations and take one descent step.

**Inputs already available:** The supplied cell creates D, target, and beta. D includes the intercept column, and its two feature columns use training-based scaling.

### Steps

1. Calculate prediction-minus-target errors as D @ beta - target. Its shape is (360,).
2. Calculate g as D.T @ error divided by len(target). This is the gradient of half MSE, not full MSE.
3. Set beta_next = beta - 0.1*g. Do not overwrite beta before the supplied before/after comparison.
4. Run the check: the gradient and new coefficients have length three, and one update reduces this training loss. Explain why one step is not the final least-squares fit.

### Commands you will use

| Command | What it does |
|---|---|
| `D @ beta` | Matrix-vector multiplication: produces one prediction per design row. |
| `D.T` | Transposes D from shape (360, 3) to (3, 360), so feature-weighted errors sum into three gradient components. |
| `len(target)` | Number of training observations; dividing the summed gradient by it forms an average. |
| `np.column_stack([ones, features])` | In the supplied setup, places a column of ones beside the scaled features. np.ones creates the ones; to_numpy converts the target Series into an array, and np.zeros initializes the coefficient vector. |
| `StandardScaler().fit(Xtr)` | In the supplied setup, learns training means and scales. Its transform method reuses them to construct D. |
| `np.mean(values ** 2)` | Squares each value and averages the results; used by the supplied loss comparison. |

**Expected result:** g and beta_next each have shape (3,). The new training squared error is smaller for this supplied step and scaled design.

**If you get stuck:** Use @, not elementwise *, for the two matrix-vector products. The formula uses prediction-minus-target; reversing that sign makes the stated update go uphill.

**Supplied task setup:** run this cell before editing the starter below.

In [ ]:
scale = StandardScaler().fit(Xtr)
D = np.column_stack([np.ones(len(Xtr)), scale.transform(Xtr)])
target = ytr.to_numpy(); beta = np.zeros(D.shape[1])

In [ ]:
# TODO 2: Complete one gradient step
# Replace each ... below, then remove the deliberate stop.
raise NotImplementedError("Complete task 2: Complete one gradient step")

error = D @ beta - target
g = ...
beta_next = ...

**Supplied check and output:** g and beta_next each have shape (3,). The new training squared error is smaller for this supplied step and scaled design. Assertions stop execution when a stated condition fails; printed scores still need interpretation.

In [ ]:
assert beta_next.shape == (3,)
assert np.mean((D@beta_next-target)**2) < np.mean((D@beta-target)**2)
print('First update:', beta_next)

## Task 3. Evaluate the fixed linear model

**Goal:** Evaluate the already fitted linear model once on the final test partition.

**Inputs already available:** Use the fitted linear and mean_baseline objects from task 1, plus Xte and yte. The task-2 beta_next is only a one-step illustration and is not the final model.

### Steps

1. Call linear.predict(Xte) and store the result in prediction.
2. Run the supplied test table comparing the baseline and linear model.
3. Read linear.coef_ in the same order as X columns: payload then queue. Explain both slopes with units.
4. Record the frozen model and both test errors. Explain why even the correct model family cannot predict the independent noise exactly.

### Commands you will use

| Command | What it does |
|---|---|
| `linear.predict(Xte)` | Returns fitted-model predictions for the 120 held-out test feature rows; it does not fit again. |
| `linear.coef_` | A learned attribute containing the two slopes. The trailing underscore marks a fitted attribute in scikit-learn. |
| `mean_baseline.predict(Xte)` | Returns the training target mean for each test row, making a comparable reference on exactly the same rows. |

**Expected result:** prediction has shape (120,). The table gives test MAE and RMSE in ms for both models, and the printed slopes retain ms/MB and ms per queued request units.

**If you get stuck:** Do not compare raw-unit slopes from linear.coef_ with scaled beta_next as if they had identical units. Do not refit using test labels.

In [ ]:
# TODO 3: Evaluate the fixed linear model
# Replace each ... below, then remove the deliberate stop.
raise NotImplementedError("Complete task 3: Evaluate the fixed linear model")

prediction = linear.predict(...)

**Supplied check and output:** prediction has shape (120,). The table gives test MAE and RMSE in ms for both models, and the printed slopes retain ms/MB and ms per queued request units. Assertions stop execution when a stated condition fails; printed scores still need interpretation.

In [ ]:
display(pd.DataFrame({'mean': regression_scores(yte, mean_baseline.predict(Xte)), 'linear': regression_scores(yte, prediction)}).T)
print('Slopes (ms/MB, ms/queued request):', linear.coef_)

## Reading the supplied outputs

The gradient check concerns one optimization step on training data. The validation/test tables concern prediction on different rows. A lower RMSE than the mean baseline is evidence on this simulation, not a guarantee that every error is that small. The code uses np.testing/assert for numerical checks and display(pd.DataFrame(...)) only to format results.

## Written interpretation

1. Interpret the two fitted slopes, explicitly stating what is held fixed and the units.
2. Explain why nonzero test error remains even when the fitted family contains the simulator’s mean relationship.
3. If the objective changes from half MSE to MSE, how do the gradient and an equivalent learning rate change?

*Write your numbered answers here.*

## Submission
Submit the completed notebook with saved outputs, numbered written answers, and a statement of the fixed or validation-selected settings. Restart the kernel and run all cells. Keep test results out of model selection; application 1 uses validation only.